In [3]:
import pandas as pd
import numpy as np

fy2020 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/LCA_Disclosure_Data_FY2020_Q4.xlsx"

df = pd.read_excel(fy2020)

print(df.shape)
print(df.head())

(117395, 96)
          CASE_NUMBER CASE_STATUS RECEIVED_DATE DECISION_DATE  \
0  I-200-20176-676243   Certified    2020-06-24    2020-07-01   
1  I-200-20176-676246   Certified    2020-06-24    2020-07-01   
2  I-200-20176-676247   Certified    2020-06-24    2020-07-01   
3  I-200-20176-676251   Certified    2020-06-24    2020-07-01   
4  I-200-20176-676254   Certified    2020-06-24    2020-07-01   

  ORIGINAL_CERT_DATE VISA_CLASS                               JOB_TITLE  \
0                NaT       H-1B                        Staff Accountant   
1                NaT       H-1B  Senior DevOps Engineer - KBGFJG72962-3   
2                NaT       H-1B  Senior DevOps Engineer - KBGFJG03640-5   
3                NaT       H-1B                      SOFTWARE ENGINEER    
4                NaT       H-1B                           SALES MANAGER   

     SOC_CODE                                         SOC_TITLE  \
0  13-2011.01                                       Accountants   
1  15-1133.

In [5]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'RECEIVED_DATE', 'DECISION_DATE', 'ORIGINAL_CERT_DATE', 'VISA_CLASS', 'JOB_TITLE', 'SOC_CODE', 'SOC_TITLE', 'FULL_TIME_POSITION', 'BEGIN_DATE', 'END_DATE', 'TOTAL_WORKER_POSITIONS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT', 'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMPLOYMENT', 'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'EMPLOYER_NAME', 'TRADE_NAME_DBA', 'EMPLOYER_ADDRESS1', 'EMPLOYER_ADDRESS2', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'NAICS_CODE', 'EMPLOYER_POC_LAST_NAME', 'EMPLOYER_POC_FIRST_NAME', 'EMPLOYER_POC_MIDDLE_NAME', 'EMPLOYER_POC_JOB_TITLE', 'EMPLOYER_POC_ADDRESS1', 'EMPLOYER_POC_ADDRESS2', 'EMPLOYER_POC_CITY', 'EMPLOYER_POC_STATE', 'EMPLOYER_POC_POSTAL_CODE', 'EMPLOYER_POC_COUNTRY', 'EMPLOYER_POC_PROVINCE', 'EMPLOYER_POC_PHONE', 'EMPLOYER_POC_PHONE_EXT', 'EMPLOYER_POC_EMAIL', 'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_LAST_NAME', 'AGENT_ATT

In [9]:
import pandas as pd

df = pd.read_excel(fy2020)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
AMAZON.COM SERVICES LLC                  2774
MICROSOFT CORPORATION                    1460
GOOGLE LLC                               1305
FACEBOOK, INC.                            972
APPLE INC.                                765
AMAZON WEB SERVICES, INC.                 543
FACEBOOK INC.                             373
TESLA, INC.                               282
AMAZON DEVELOPMENT CENTER U.S., INC.      226
NVIDIA CORPORATION                        145
AMAZON WEB SERVICES INC                   109
ADVANCED MICRO DEVICES, INC.               63
AMAZON.COM SERVICES LLC.                   60
MICRON TECHNOLOGY, INC.                    51
AMAZON DATA SERVICES, INC.                 49
MICRONET IT SOLUTIONS, INC.                37
METASOFTTECH SOLUTIONS, LLC.               14
AMAZON WEB SERVICES, INC                   12
AMAZON ROBOTICS LLC                        10
METAPRO USA, LLC                            7
SYNMETA LLC                                 6
AMAZON ADVERTISING L

In [11]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Amazon       2774
Microsoft    1460
Alphabet     1305
Meta          972
Apple         765
Tesla         282
NVIDIA        145
AMD            63
Micron         51
Name: count, dtype: int64


In [13]:
df_clean = df.copy()

In [15]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY"] = (
    df["WORKSITE_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Seattle     1902
New York     320
Austin       207
San Jose      20
Dallas         8
Name: count, dtype: int64


In [17]:
print(df["WAGE_UNIT_OF_PAY"].value_counts())

WAGE_UNIT_OF_PAY
Year    2454
Hour       3
Name: count, dtype: int64


In [19]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"]
    * df["WAGE_UNIT_OF_PAY"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]].head(10))

    Company      City  WAGE_RATE_OF_PAY_FROM WAGE_UNIT_OF_PAY  Annual_Wage
44   Amazon   Seattle               155300.0             YEAR     155300.0
100  Amazon   Seattle               160000.0             YEAR     160000.0
101  Amazon   Seattle               160000.0             YEAR     160000.0
102  Amazon   Seattle               138500.0             YEAR     138500.0
103  Amazon    Dallas               103200.0             YEAR     103200.0
108  Amazon   Seattle               125000.0             YEAR     125000.0
111  Amazon   Seattle               154250.0             YEAR     154250.0
116  Amazon   Seattle               155000.0             YEAR     155000.0
120  Amazon   Seattle               160000.0             YEAR     160000.0
123  Amazon  New York               174800.0             YEAR     174800.0


In [21]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count    2.457000e+03
mean     1.423057e+05
std      3.715367e+04
min      5.500000e+04
25%      1.237600e+05
50%      1.440000e+05
75%      1.572000e+05
max      1.430000e+06
Name: Annual_Wage, dtype: float64


In [23]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [25]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                 29     128606.0  134960.075862
1     Austin   Alphabet                 18     130000.0  124444.333333
2     Austin     Amazon                 63     144800.0  138191.603175
3     Austin      Apple                 61     145000.0  148753.721311
4     Austin       Meta                 22     155000.0  148664.243636
5     Austin  Microsoft                  2     124000.0  124000.000000
6     Austin     NVIDIA                  8     135447.0  149132.050000
7     Austin      Tesla                  4     142020.0  143650.000000
8     Dallas     Amazon                  6     143650.0  137583.333333
9     Dallas      Apple                  2     167836.5  167836.500000
10  New York   Alphabet                125     152000.0  168072.856000
11  New York     Amazon                 91     160600.0  153880.912088
12  New York      Apple                  3     215000.0  198333.333333
13  Ne

In [27]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_TITLE",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2020.csv",
    index=False
)